In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import gc
from collections import defaultdict
from cycler import cycler
from tools import manhattan_plot, downsample_gwas

!pip install -q gseapy
!pip install -q cyvcf2
from cyvcf2 import VCF
import gseapy as gp
import networkx as nx
import seaborn as sns

# Table

In [ ]:
df_fisher = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/CN-LOH.GWAS.FDR.01.txt', sep='\t')
df_phase = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/refined_phase/FDR.01.burden.allelic_shift.GWAS.txt.gz', sep='\t')
df_fisher \
    .astype({'pos': np.int32}) \
    .rename({'pos': 'Mb', 'pval': 'fisher_pval'}, axis = 1) \
    .assign(merge = lambda x: x['gene'] + '.' + x['mask']) \
    .merge(df_phase.rename({'mask':'merge'}, axis=1)) \
    .drop(['merge', 'threshold', 'missing_phase', 'inconsistent_phase'], axis=1) \
    .assign(
        bayesian_two_sided_tail_prob = lambda x: 2*np.minimum(
            scipy.stats.beta(1/2 + x['carrier_overrep'], 1/2+x['carrier_underrep']).cdf(0.5),
            scipy.stats.beta(1/2 + x['carrier_overrep'], 1/2+x['carrier_underrep']).sf(0.5)   
        )
    ) \
    .to_csv('burden_gwas_table.csv', index=False) 
gc.collect()

# CF

In [ ]:
df_cf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/ID.cf.gene.txt', sep='\t')
df_cf['x'] = pd.factorize(df_cf['gene'])[0]

df_hmm = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
mean_cf = df_hmm.query('type=="CN-LOH"')['cf'].median()
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
ax.scatter(df_cf['x']+np.random.normal(0, 0.1, size=df_cf['x'].shape), df_cf['cf'], marker='.')
df_cf_summary = df_cf.groupby('gene').agg(
    x = ('x', 'first'),
    median_cf = ('cf', 'median'),
    upper_quartile_cf = ('cf', lambda x: np.percentile(x, 75)),
    lower_quartile_cf = ('cf', lambda x: np.percentile(x, 25)),
    n = ('cf', 'size')
)
ax.errorbar(df_cf_summary['x'], df_cf_summary['median_cf'], yerr=[
    df_cf_summary['median_cf'] - df_cf_summary['lower_quartile_cf'],
    df_cf_summary['upper_quartile_cf'] - df_cf_summary['median_cf']
], fmt='o', color='red', ecolor='black', elinewidth=2, capsize=4)
ax.set_xticks((df_cf_summary['x']), [r'$%s$' % i for i in df_cf_summary.index], rotation=90)
ax.tick_params(axis='both', labelsize=14)
ax.set_ylabel('CN-LOH cell fraction', fontsize=16)
ax.set_xlabel('Gene', fontsize=16)
ax.axhline(mean_cf, color='grey', linestyle='--', label='Median CN-LOH cell fraction')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=14)

plt.savefig('cnloh_burden_carrier_cf.pdf', transparent=True, bbox_inches='tight')

In [ ]:
import warnings
df_carriers = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/ID.mask.txt', sep='\t', header=None)
df_carriers.columns = ['ID', 'gene', 'mask']
df_blood = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/not_restricted_to_analyzed_samples/ID.blood_counts.txt', sep='\t')
high_cf_blood_counts = df_cf \
    .merge(df_carriers, on=['ID', 'gene'], how='outer')[['ID', 'cf', 'gene', ]] \
    .merge(df_blood) \
    .query('(cf>0.05 or cf.isna()) and (gene == "MPL" or gene == "JAK2" or gene == "ATM" or gene=="TM2D3")') 
df_long = pd.concat(
    [
        high_cf_blood_counts.query('cf.isna()').groupby('gene', group_keys=False).apply(lambda x: x.sample(100, random_state=42)),
        high_cf_blood_counts.query('cf>0.05')
    ]
).melt(
    id_vars=['gene', 'cf'],
    value_vars=['lymphocyte', 'monocyte', 'neutrophill', 'eosinophill', 'basophill'],
    var_name='cell_type',
    value_name='count'
).assign(
    cnloh = lambda x: np.where(x['cf'].isna(), 'No CN-LOH', 'CN-LOH')
)

for gene in ['JAK2', 'MPL', 'ATM', 'TM2D3']:
    fig, ax = plt.subplots(dpi=150)
    ax.set_ylim(0, 22)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*cannot be placed.*")
        sns.swarmplot(
            df_long.query('gene==@gene and count<20'),
            x='cell_type',
            y='count',
            hue='cnloh',
            size=2,
            ax=ax,
            palette=['gray', 'gold'],
            dodge=True
        )
    for ct in ['lymphocyte', 'monocyte', 'neutrophill', 'eosinophill', 'basophill']:
        ct_cnloh = high_cf_blood_counts.query('gene==@gene and cf.notna()')[ct]
        ct_control = high_cf_blood_counts.query('gene==@gene and cf.isna()')[ct]
        stat, pval = scipy.stats.mannwhitneyu(ct_cnloh, ct_control, alternative='two-sided')
        y = df_long.query('gene==@gene and cell_type==@ct and count < 20')['count'].max()+1
        if pval < 0.05:
            ax.annotate(f'p = {pval:.1e}', xy=(ct, y), xytext=(ct, y), ha='center', fontsize=10)
        print(f'{gene} {ct} p={pval:.2e} (n={ct_cnloh.shape[0]} vs n={ct_control.shape[0]})') 

    ax.legend(title=rf'$\it{{{gene}}}$ rare variant carriers', frameon=False, fontsize=10, markerscale=2, title_fontsize=10)
    ax.set_ylabel('Cell count (cells/nL)', fontsize=12)
    ax.set_xlabel('Cell type', fontsize=12)

# Phase

In [ ]:
df_sig = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/refined_phase/FDR.01.burden.allelic_shift.GWAS.txt.gz', sep='\t')
df_sig['gene'] = [x.split('.')[0] for x in df_sig['mask']]

df_phase = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/refined_phase/CN-LOH.burden.allelic_shift.GWAS.txt.gz', sep='\t')
sig_genes = set(df_sig['gene'])
df_phase['gene'] = [x.split('.')[0] for x in df_phase['mask']]
df_phase['sig'] = [x in sig_genes for x in df_phase['gene']]
depmap = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/refined_phase/allelic_shift.depmap.txt', sep='\t')
df_phase = df_phase \
    .merge(depmap, on='gene', how='outer') \
    .drop(['var_hom', 'var_removed', 'shift_direction'], axis = 1)
df_phase['quantile_lymphoid_depmap'] = pd.qcut(df_phase['lymphoid_depmap'], 10, labels=False)
df_phase['quantile_myeloid_depmap'] = pd.qcut(df_phase['myeloid_depmap'], 10, labels=False)

In [ ]:
def plot_horizontal_bar(y, df, ax):
    height = 0.9
    c1 = '#16d051d0'
    c2 = '#c86bdf'
    var_hom = (df['carrier_overrep'] > df['carrier_underrep']).sum()
    var_removed = (df['carrier_overrep'] < df['carrier_underrep']).sum()
    total = var_hom + var_removed
    lower = scipy.stats.beta(1/2+var_removed, 1/2+var_hom).ppf(0.025) 
    upper = scipy.stats.beta(1/2+var_removed, 1/2+var_hom).ppf(0.975)
    print(var_hom, var_removed, total)
    if ((lower-0.5) * (upper-0.5)) > 0:
        ax.text(x=upper+0.01, y=y, s='*', ha='left', va='center_baseline')
    ax.barh(y, var_removed/total, height=height, color=c1, xerr=([var_removed/total - lower], [upper - var_removed/total]), capsize=10)
    ax.barh(y, var_hom/total, left=var_removed/total, height=height, color=c2)

fig, ax = plt.subplots(dpi=600, figsize=(6, 3))
plot_horizontal_bar(0, df_phase.query('sig'), ax)
plot_horizontal_bar(-1, df_phase.query('quantile_lymphoid_depmap == 0'), ax)
plot_horizontal_bar(-2, df_phase.query('oncogene==0 and tsg==1'), ax)
plot_horizontal_bar(-3, df_phase.query('oncogene==1 and tsg==0'), ax)

ax.axvline(0.5, color='k', linestyle=':')

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.set_yticks([-3, -2, -1, 0], ['Oncogenes', 'TSGs', 'DepMap.Q10', 'CN-LOH associated'], fontsize=14)
ax.set_xticks(np.arange(0, 1+0.1, 0.1))
ax.set_xlim(0, 1)

ax.set_xlabel('Proportion of genes for which CN-LOH \n more often removed damaging variation', fontsize=14)
plt.savefig('shift_direction.pdf', transparent=True, bbox_inches='tight')
plt.show()


# Pathways

In [ ]:
# Perform GO enrichment analysis
enrichment_results = gp.enrichr(gene_list=list(sig_genes),
                                gene_sets=['GO_Biological_Process_2025', 'KEGG_2021_Human', 'Reactome_Pathways_2024'],  # Choose GO category
                                organism='Human',
                                outdir=None)  # Set a directory to save results if needed

# Convert results to DataFrame
df_enr = enrichment_results.results
df_enr['num_genes'] = [int(x.split('/')[0]) for x in df_enr['Overlap']]
df_enr['pval']  = df_enr['P-value'].astype(float)

In [ ]:
from IPython.display import display
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_enr.query('pval<1e-3').head(10))

In [ ]:
df_enr[df_enr['Adjusted P-value'] <0.05].drop(['num_genes', 'pval', 'Old P-value', 'Old Adjusted P-value'], axis = 1).to_csv('enrichment_results.csv', index=False)

In [ ]:
c1 = '#16d051d0'
c2 = '#c86bdf'
# Create graph
G = nx.Graph()

terms = {
    'DNA Double Strand Break Response',
    'tRNA Wobble Uridine Modification',
    'TM2 domain',
    'Cytokine Signaling in Immune System',
    'Thrombopoietin-Mediated Signaling Pathway',
    'Mitotic G2/M Transition Checkpoint',
    'Regulation of Apoptotic Process',
    'Protein Ubiquitination',
}

# Build graph: connect terms to their genes
for _, row in df_enr.iterrows():
    if row['P-value'] > 1e-3: continue
    term = row['Term']
    if row['Gene_set'] == 'GO_Biological_Process_2025':
        term_name = ' '.join(term.split(' ')[:-1])
    else:
        term_name = term
    if term_name not in terms: continue
    genes = row['Genes'].split(';')  # genes are semicolon-separated

    for gene in genes:
        G.add_node(gene, type='gene')
        G.add_node(term_name, type='term')
        G.add_edge(term_name, gene)

for gene in ['TM2D1', 'TM2D2', 'TM2D3']:
    G.add_node(gene, type='gene')
    G.add_node('TM2 domain', type='protein_domain')
    G.add_edge('TM2 domain', gene)

# Draw network
fig, ax = plt.subplots(figsize=(14, 9), dpi=150)
gene_nodes = [n for n, attr in G.nodes(data=True) if attr['type'] == 'gene']
df_sig['two_sided_tail_prob'] = 2*np.minimum(
    scipy.stats.beta(1/2+df_sig['carrier_overrep'], 1/2+df_sig['carrier_underrep']).cdf(0.5),
    scipy.stats.beta(1/2+df_sig['carrier_overrep'], 1/2+df_sig['carrier_underrep']).sf(0.5)
)
gray_genes = set(df_sig[(df_sig['two_sided_tail_prob'] > 0.1)]['gene'])
purple_genes = set(df_sig[df_sig['carrier_overrep'] > df_sig['carrier_underrep']]['gene']).difference(gray_genes)
green_genes =  set(df_sig[df_sig['carrier_overrep'] < df_sig['carrier_underrep']]['gene']).difference(gray_genes)
node_colors = [
    c1 if x in green_genes else
    c2 if x in purple_genes else
    'lightgray' 
    for x in gene_nodes
]
pos = nx.spring_layout(G, seed=2, k=1/2.8)
pos['Thrombopoietin-Mediated Signaling Pathway'] += np.array([0, -0.05]) 
pos['MPL'] += np.array([-0.2, 0.1]) 
pos['Mitotic G2/M Transition Checkpoint'] += np.array([-0, -0.04])
pos['DNA Double Strand Break Response'] += np.array([0.2, 0.05])
pos['ATM'] += np.array([0.2, -0.2])
pos['Cytokine Signaling in Immune System'] += np.array([-0.1, 0.1])
pos['TP53'] += np.array([0.2, 0.06])
pos['CHEK2'] += np.array([0.05, 0])
pos['TM2 domain'] += np.array([-0.1, 0.2])
pos['TM2D2'] += np.array([-0.1, 0.15])
pos['TM2D3'] += np.array([0., 0.25])
pos['TM2D1'] += np.array([-0.15, 0.1])
pos['URM1'] += np.array([0.12, 0.05])
nx.draw_networkx_nodes(G, pos,
                       nodelist=gene_nodes,
                       node_shape='o',
                       node_color=node_colors,
                    #    edgecolors='k',
                       node_size=1700,
                       label='Genes')

nx.draw_networkx_edges(G, pos, edge_color='gray')
labels = {node: rf'$\it{{{node}}}$' for node in gene_nodes}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=12, ax=ax)

for i,term in enumerate(terms):
    ax.text(
        pos[term][0], 
        pos[term][1], 
        term, 
        fontsize=12, 
        ha='center', 
        va='center', 
        bbox=dict(facecolor='lightcoral', alpha=1)
    )

for spine in ['bottom', 'left', 'top', 'right']:
    ax.spines[spine].set_visible(False)
plt.savefig('GO_enrichment_network.pdf', transparent=True, bbox_inches='tight')

# Penetrance

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/CN-LOH.GWAS.txt.gz', sep='\t').query(
    'ref=="ref" or variant=="9:5073770:G:T" or  variant=="1:241512001:G:C"'
)
df['gene'] = [
    x.split('.')[0] 
    if x!='9:5073770:G:T' and x!="1:241512001:G:C" 
    else 'JAK2:V617F' if x=='9:5073770:G:T'
    else 'FH:P174R' for x in  df['variant']
]
df = df.query('gene in @sig_genes or variant=="9:5073770:G:T" or  variant=="1:241512001:G:C"')
gc.collect()

In [ ]:
chromosome_centromeres_GRCh38 = {
    "chr1": 122026459,
    "chr2": 92188145,
    "chr3": 90772445,
    "chr4": 50000000,
    "chr5": 48800000,
    "chr6": 59800000,
    "chr7": 61000000,
    "chr8": 45500000,
    "chr9": 49000000,
    "chr10": 40200000,
    "chr11": 53700000,
    "chr12": 35800000,
    "chr13": 17900000,
    "chr14": 17600000,
    "chr15": 19000000,
    "chr16": 36800000,
    "chr17": 25100000,
    "chr18": 18500000,
    "chr19": 27600000,
    "chr20": 29300000,
    "chr21": 13200000,
    "chr22": 14700000,
    "chrX": 58600000,
    "chrY": 10100000
}

blacklist = set('PTPRF|MMACHC|SPRYD7|DLEU7|KCNRC|RNASEH2B|TRIM13|OR4F15|OR4F6|TARS3|JAK2|FH|KCNRG|KPAN3'.split('|'))
df['chrom_arm'] = [
    x['chr']+ ('.p' if x['pos'] < chromosome_centromeres_GRCh38[x['chr']] else '.q') 
    for _,x in df.iterrows()
]
df = df.query('not gene in @blacklist')

In [ ]:
df_penetrance = df.query('alt=="LoF.all_transcripts.CNV.0.001"')
variants_to_extract = {
    var:f'{chrom}:{pos}-{pos}' 
    for chrom, pos, var in zip(
        df_penetrance['chr'], df_penetrance['pos'], df_penetrance['variant']
    )
}
chrom_to_variants_to_extract = {
    chrom: {
        var: region 
        for var, region in variants_to_extract.items() if region.split(':')[0] == chrom
    }
    for chrom in [f'chr{c}' for c in range(1, 22+1)]
}

gt = []
for chrom in chrom_to_variants_to_extract.keys():
    print(chrom)
    vcf = VCF(f'/mnt/project/lohdata/resources/burden_masks/vcfs/{chrom}.LoF.scaffold.phased.AS.bcf')
    samples = np.array([int(x) for x in vcf.samples])
    for var, region in chrom_to_variants_to_extract[chrom].items():
        pos = int(region.split(':')[1].split('-')[0])
        for record in vcf(region):
            if record.ID == var: 
                for sample in samples[(record.gt_types==1)]:
                    gt.append([sample, var, chrom, pos])
                print(record.ID)
                break
    gc.collect()
gt = np.array(gt)

In [ ]:
df_vaf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/cisGWAS/results/LoF.all_transcripts.0.001.VAF.txt.gz', sep='\t')
df_vaf = df_vaf.groupby('gene').agg(
    refAD = ('refAD', 'sum'),
    altAD = ('altAD', 'sum'),
)
df_vaf['lower_ci'] = scipy.stats.beta(df_vaf['altAD']+1/2, df_vaf['refAD']+1/2).ppf(0.005)
df_vaf['upper_ci'] = scipy.stats.beta(df_vaf['altAD']+1/2, df_vaf['refAD']+1/2).ppf(0.995)
df_vaf['frac_alt'] = df_vaf['altAD'] / (df_vaf['altAD'] + df_vaf['refAD'])

df_gt = pd.DataFrame(gt, columns=['ID', 'variant', 'chr', 'pos']).astype({
    'ID': 'int32',
    'variant': 'string',
    'chr': 'string',
    'pos': 'int32'
})
df_age = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID_age.40709.txt', sep='\t')
df_age['ageBin']  = df_age['age']//5 *5
somatic_qc = df_gt.merge(df_age).assign(gene=lambda x: [v.split('.')[0] for v in x['variant']]).groupby('gene').agg(
    mean_age = ('age', 'mean'),
    num_carriers = ('ID', 'nunique'),
    std_age = ('age', 'std')
).assign(
    sem_age = lambda x: x['std_age'] / np.sqrt(x['num_carriers'])
).merge(df_vaf, on='gene').query('num_carriers>30').sort_values(by = 'frac_alt').reset_index()

germline_genes = set(somatic_qc.query('upper_ci > 0.5')['gene'].tolist() + ['CFLAR'])


fig,ax = plt.subplots(2, 1, figsize=(8, 8), dpi=150)
for idx, row in somatic_qc.iterrows():
    ax[0].errorbar(
        idx,
        row['mean_age'],
        yerr=row['sem_age']*scipy.stats.norm.ppf(0.995),
        fmt='o',
        capsize=5,
        color = 'gray' if row['gene'] in germline_genes else 'red'
    )
ax[0].axhline(df_age['age'].mean(), color='k', linestyle=':')
ax[0].set_xticks([])
ax[0].set_ylabel('Mean age of carriers', fontsize=14)
ax[0].spines['right'].set_visible(False)
ax[0].spines['top'].set_visible(False)
ax[0].spines['bottom'].set_visible(False)

for idx,row in somatic_qc.iterrows():
    ax[1].errorbar(
        idx, 
        row['frac_alt'], 
        yerr=[[row['frac_alt']-row['lower_ci']], 
            [row['upper_ci']-row['frac_alt']]], 
        fmt='o', 
        capsize=5,
        color = 'gray' if row['gene'] in germline_genes else 'red'
    )
ax[1].set_xticks(somatic_qc.index, [rf'$\it{{{gene}}}$' for gene in somatic_qc['gene']], rotation=90)
ax[1].axhline(0.5, color='k', linestyle=':')
ax[1].set_ylabel('Fraction of reads from LoF allele', fontsize=14)
ax[1].spines['right'].set_visible(False)
ax[1].spines['top'].set_visible(False)
ax[1].spines['bottom'].set_visible(False)

ax[0].text(-0.15, 1.15, 'a', transform=ax[0].transAxes, fontsize=16, va='top', ha='right')
ax[1].text(-0.15, 1.15, 'b', transform=ax[1].transAxes, fontsize=16, va='top', ha='right')
plt.savefig('somatic_qc_CNLOH_burden.pdf', transparent=True, bbox_inches='tight')


In [ ]:
calls = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
calls = calls.assign(
    bpStart = lambda x: np.where(x['p']=="T", 0, x['bpStart']),
    bpEnd = lambda x: np.where(x['q']=="T", 1e9, x['bpEnd'])
)

df_penetrance = calls.query('type=="CN-LOH"').merge(df_gt, on=['ID', 'chr'], how='right').assign(
    keep = lambda x: (x['pos'] > x['bpStart']) & (x['pos'] < x['bpEnd'])
).groupby(['ID', 'variant']).agg(
    keep = ('keep', 'sum')
).assign(
    mCA = lambda x: x['keep']>0,
).groupby(
    ['variant']
).agg(
    carrier_mCA = ('mCA', 'sum'),
    total_carriar = ('mCA', 'count')
).assign(
    carrier_control = lambda x: x['total_carriar'] - x['carrier_mCA'],
).reset_index().query('total_carriar >= 30')

df_penetrance['penetrance'] = df_penetrance['carrier_mCA']/(df_penetrance['carrier_mCA']+ df_penetrance['carrier_control'])
df_penetrance['lower_penetrance'] = [
    scipy.stats.beta(x['carrier_mCA']+1/2, x['carrier_control']+1/2).ppf(0.025) 
    for _,x in df_penetrance.iterrows()
]
df_penetrance['upper_penetrance'] = [
    scipy.stats.beta(x['carrier_mCA']+1/2, x['carrier_control']+1/2).ppf(0.975) 
    for _,x in df_penetrance.iterrows()
]
df_penetrance['penetrance'] = df_penetrance['carrier_mCA']/(df_penetrance['carrier_control']+df_penetrance['carrier_mCA'])
df_penetrance = df_penetrance.sort_values('penetrance', ascending=False).reset_index(drop=True)
df_penetrance['gene'] = [
    x.split('.')[0] 
    for x in  df_penetrance['variant']
]
df_penetrance = df_penetrance.query('gene in @germline_genes').reset_index(drop=True)

fig, ax = plt.subplots(dpi=150, figsize=(8,4))
ax.bar(
    df_penetrance.index, 
    df_penetrance['penetrance'], 
    yerr=(
        df_penetrance['penetrance']-np.minimum(df_penetrance['lower_penetrance'], df_penetrance['penetrance']), 
        df_penetrance['upper_penetrance']-df_penetrance['penetrance']
    ),
    capsize=2,
    edgecolor='k',
    color = 'peachpuff'
)
ax.set_xticks(
    df_penetrance.index, 
    [rf'$\it{{{gene}}}$' for gene in df_penetrance['gene']], 
    rotation=90
)
ax.set_ylabel('Penetrance of LoFs on\nCN-LOH clonal expansion', fontsize=16)
ax.set_xlabel('Gene', fontsize=16)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlim(0-0.6, len(df_penetrance))
ax.tick_params('y', labelsize=12)



from mpl_toolkits.axes_grid1.inset_locator import inset_axes
inset_ax = inset_axes(ax, width="50%", height="60%", loc='upper right')


df_ind = calls \
    .query('type=="CN-LOH"') \
    .merge(df_gt, on=['ID', 'chr'], how='right') \
    .assign(
        keep = lambda x: (x['pos'] > x['bpStart']) & (x['pos'] < x['bpEnd'])
    ) \
    .groupby(['ID', 'variant']).agg(
        keep = ('keep', 'sum')
    ).reset_index() \
    .assign(
        mCA = lambda x: x['keep']>0,
    ) \
    .merge(df_age, on='ID', how='left') \
    .assign(
        gene = lambda x: x['variant'].apply(lambda y: y.split('.')[0] if y!='9:5073770:G:T' and y!="1:241512001:G:C" else 'JAK2:V617F' if y=='9:5073770:G:T' else 'FH:P174R'),
    ) \
    .query('age < 70 and age >=40') \
    .drop(['variant', 'keep'], axis=1) 

df_plot = df_ind.groupby('ageBin').agg(
    a = ('mCA', 'sum'),
    total = ('mCA', 'count')
).reset_index() \
    .assign(
        b = lambda x: x['total'] - x['a']
    ) \
    .assign(
    penetrance = lambda x: x['a']/(x['a']+x['b']),
    lower_penetrance = lambda x: x['a']/(x['a']+x['b']) - scipy.stats.beta(x['a']+1/2, x['b']+1/2).ppf(0.025),
    upper_penetrance = lambda x: scipy.stats.beta(x['a']+1/2, x['b']+1/2).ppf(0.975) - x['a']/(x['a']+x['b'])
)

inset_ax.errorbar(
    df_plot['ageBin'], 
    df_plot['penetrance'], 
    yerr=(
        df_plot['lower_penetrance'], 
        df_plot['upper_penetrance']
    ), 
    color='k',
    capsize=5, 
    marker='.',
    linestyle='',
    label = 'All genes'
)

inset_ax.set_xlabel('Age (years)', fontsize=16)
inset_ax.set_ylabel('Penetrance', fontsize=16)

offset = 1
for gene, group in df_ind.groupby(['ageBin', 'gene']).agg(
    a = ('mCA', 'sum'),
    total = ('mCA', 'count')
).reset_index().groupby('gene'):
    if not gene in {'TM2D3', 'MPL', 'ATM'}: continue
    a = group['a']
    b = group['total'] - a
    lower = a/(a+b) - scipy.stats.beta(a+1/2, b+1/2).ppf(0.025)
    upper = scipy.stats.beta(a+1/2, b+1/2).ppf(0.975) - a/(a+b)
    print(gene)
    print(pd.concat([a/(a+b)-lower,a/(a+b), upper+a/(a+b)], axis = 1))
    inset_ax.errorbar(
        group['ageBin']+offset*0.5, 
        a/(a+b), 
        yerr=(
            lower, 
            upper
        ), 
        marker='.',
        linestyle='',
        label = rf'$\it{{{gene}}}$',
        capsize=5,
    )
    offset += 1
    print()

inset_ax.set_xlabel('Age (years)', fontsize=12)
inset_ax.set_ylabel('LoF penetrance', fontsize=12)
inset_ax.grid(axis='y', linestyle=':', alpha=0.5)


inset_ax.plot(color='k', marker='o', alpha=0.5, label='95% CI')
inset_ax.legend(frameon=False, fontsize=8, ncol=2)
inset_ax.set_ylim(0, 1)
inset_ax.set_yticks(np.arange(0, 1+0.2, 0.2))
inset_ax.set_xticks(
    np.array([40, 45, 50, 55, 60, 65]) + 1.5*0.5, 
    ['40-44', '45-49', '50-54', '55-59', '60-64', '65-69']
)
plt.savefig('burden_test_penetrance.pdf', bbox_inches='tight', transparent=True)

# Fraction explained

In [ ]:
calls = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep='\t')
num_calls = defaultdict(int)
high_cf_calls = defaultdict(int)
seen = set()
for _,row in calls.query('type=="CN-LOH"').iterrows():
    if(row['p']!='N'): 
        chr_arm = row['chr']+'.p'
        if(not (row['ID'], chr_arm) in seen): num_calls[chr_arm] += 1
        seen.add((row['ID'], chr_arm))
        if row['cf'] > 0.1: high_cf_calls[row['chr']+'.p'] += 1
    if(row['q']!='N'): 
        chr_arm = row['chr']+'.q'
        if(not (row['ID'], chr_arm) in seen): num_calls[chr_arm] += 1
        seen.add((row['ID'], chr_arm))
        if row['cf'] > 0.1: high_cf_calls[row['chr']+'.q'] += 1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_chrom_arm_explained = df.sort_values('carrier_mCA', ascending=False).drop_duplicates('gene', keep='first')
new_colors = ['#E69F00', '#F4C430', '#FFD166', '#FFEB99', '#FFF5B2', '#FAF0CA']


width = 1
for i in range(22*2):
    chrom = i//2 + 1
    arm = 'p' if i % 2 == 0 else 'q'
    chrom_arm = f'chr{chrom}.{arm}'
    bottom = 0
    ax.set_prop_cycle(cycler(color=new_colors))
    for _, row in df_chrom_arm_explained.query('chrom_arm == @chrom_arm').iterrows():
        explained_mCAs = row['carrier_mCA']/num_calls[chrom_arm]
        color = 'peachpuff' if chrom % 2 == 0 else 'beige'
        ax.bar(i, explained_mCAs, bottom=bottom, edgecolor='k', linewidth=1.5, width=1, align='edge')
        bottom+=explained_mCAs
ax.set_xticks(np.arange(0, 22*2, 2)+1, [f'{chrom}' for chrom in range(1,22+1)], rotation=90)
ax.set_ylabel('Fraction of CN-LOH driven\nby rare coding variant', fontsize=16)
ax.set_xlabel('Chromosome', fontsize=16)
ax.tick_params(axis='both', labelsize=16)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

gene_size=12
ax.text(x=1*2-1, y=0.24, s='$\it{MPL}$', fontsize=gene_size)
ax.text(x=9*2-2, y=0.24, s='$\it{JAK2}$', fontsize=gene_size, ha='right')
ax.text(x=11*2, y=0.24, s='$\it{ATM}$', fontsize=gene_size)
ax.text(x=15*2, y=0.32, s='$\it{TM2D3}$', fontsize=gene_size)
ax.text(x=12*2, y=0.13, s='$\it{SH2B3}$', fontsize=gene_size)
ax.text(x=4*2, y=0.08, s='$\it{TET2}$', fontsize=gene_size)
ax.set_yticks(np.arange(0, 0.4, 0.05))
plt.savefig('CNLOH_explained_by_rare_var.pdf', bbox_inches='tight')
plt.show()

In [ ]:
num_calls_explained_by_rare_var = defaultdict(
    int, 
    df_chrom_arm_explained.groupby('chrom_arm').agg(
        carrier_mCA = ('carrier_mCA', 'sum')
    ).to_dict()['carrier_mCA']
)
numer = 0
denom = 0
for chrom_arm, num in num_calls.items():
    numer += num_calls_explained_by_rare_var[chrom_arm]
    denom += num_calls[chrom_arm]
print(f'Fraction of CN-LOH calls explained by rare coding variants: {numer/denom:.2%} ({numer}/{denom})')